# AwA2 Model Testing — Cross-Dataset Transfer

**Compositional Image Retrieval — Deep Learning Assignment 2026 (secondary benchmark)**

This notebook is a *secondary* study: it re-runs our full CelebA model roster on
**Animals with Attributes 2** (AwA2, spec §2) to probe how each method transfers to a
dataset with a very different attribute structure. It is deliberately compact — the main
project deliverable is the CelebA benchmark. The one thing it explores in depth is the
*surprise*: the ranking of methods **inverts** relative to CelebA, and understanding why
is what makes this a useful experiment.

The evaluation protocol is identical to CelebA (frozen CLIP ViT-B/32 DB, spec §3.1.1
relaxed-Hamming ground truth, Recall/Precision@{1,5,10}, stratified bootstrap CIs and
paired significance from `src/stats_eval.py`). Only the dataset and the 12-query
benchmark (`Evaluation/awa2_evaluation.json`) change.

**Models under comparison** (display name ← AwA2 run)

| Display name | Method | Type | direction source |
|---|---|---|---|
| **LDE** | mined visual directions, flat arithmetic | training-free | visual |
| **GDE** | mined visual directions, geodesic + rejection | training-free | visual |
| **CLIP Enhanced** | text deltas, all geometry fixes (tier-0) | training-free | text |
| **DGP** | gated projection, uniform gate (tier-2d) | training-free | text |
| **Dual Score** | query-conditioned dual-channel fusion (tier-3) | learned | text + probe |
| **CLAY** | pre-SVD prompt stacking (k=50, rotH) | training-free | text subspace |
| **A-BAP** | CD-Bayes + learned Φ (tier-5) — *CelebA champion* | learned | probe |

> **A-BAP** is the display name of our CelebA champion (`tier5_Bprime_listnet`): an
> Amortized Bayesian Attribute-Posterior fusion. On CelebA it wins decisively; on AwA2,
> as we show below, it finishes **last**.


## 1. Setup

Thin cells only — all statistics live in `src/stats_eval.py`, the per-source dumps in
`src/per_source.py` (each parity-gated against its published CSV). Any model whose
per-source dump is not present locally (e.g. a VM-trained one not yet pulled) is skipped
with a message and simply omitted from the figures.

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd().parent if (Path.cwd() / "..").resolve().name == "Deep-Learning-Project" else Path.cwd()
ROOT = ROOT if (ROOT / "src").exists() else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from eval import find_eval_json, load_eval_json
from per_source import load_per_source
from stats_eval import (holm_bonferroni, metric_label, paired_bootstrap_diff,
                        plot_ci_bars, recall_curve, stratified_bootstrap)

B = 1000
GT_LIST = load_eval_json(find_eval_json().parent / "awa2_evaluation.json")
PS = ROOT / "output" / "awa2" / "per_source"

# Ordered training-free (visual) → training-free (text) → learned, so the figure legend
# reads top-to-bottom in the ranking we discuss.
MODELS = {
    "LDE":           "awa2_tier_lde.npz",
    "GDE":           "awa2_tier_gde.npz",
    "CLIP Enhanced": "awa2_tier0_baseline.npz",
    "DGP":           "awa2_tier2d_tau100.0_a1.0_center.npz",
    "Dual Score":    "awa2_tier3_dualscore_seed0.npz",
    "CLAY":          "awa2_tier_clay.npz",
    "A-BAP":         "awa2_tier5_bprime_listnet.npz",
}

dumps = {}
for name, fname in MODELS.items():
    p = PS / fname
    if p.exists():
        dumps[name] = load_per_source(p)
    else:
        print(f"[missing] {fname} — '{name}' skipped (pull its per-source NPZ from the VM).")
print("loaded:", list(dumps))

[missing] awa2_tier3_dualscore_seed0.npz — 'Dual Score' skipped (pull its per-source NPZ from the VM).
loaded: ['LDE', 'GDE', 'CLIP Enhanced', 'DGP', 'CLAY', 'A-BAP']


## 2. The result — and the inversion

Point estimates with bootstrap 95% CIs for every model × spec metric (`stratified_bootstrap`,
§A.3 of the CelebA statistical notebook — same code).

In [2]:
boot = {name: stratified_bootstrap(d, B=B, seed=0) for name, d in dumps.items()}
rows = []
for name, res in boot.items():
    row = {"model": name}
    for f, r in res.items():
        row[metric_label(f)] = f"{r.point:.4f}  [{r.ci_lo:.4f}, {r.ci_hi:.4f}]"
    rows.append(row)
pd.DataFrame(rows).set_index("model")

,R@1,R@5,R@10,P@1,P@5,P@10
model,,,,,,
LDE,"0.8806 [0.8569, 0.9028]","0.9722 [0.9597, 0.9833]","0.9847 [0.9750, 0.9931]","0.8806 [0.8569, 0.9028]","0.8511 [0.8322, 0.8683]","0.8360 [0.8186, 0.8531]"
GDE,"0.8681 [0.8458, 0.8917]","0.9597 [0.9458, 0.9722]","0.9750 [0.9639, 0.9847]","0.8681 [0.8458, 0.8917]","0.8322 [0.8144, 0.8503]","0.8132 [0.7957, 0.8308]"
CLIP Enhanced,"0.8083 [0.7805, 0.8361]","0.9361 [0.9194, 0.9528]","0.9639 [0.9500, 0.9778]","0.8083 [0.7805, 0.8361]","0.7761 [0.7536, 0.7989]","0.7528 [0.7299, 0.7746]"
DGP,"0.7653 [0.7361, 0.7958]","0.9292 [0.9097, 0.9458]","0.9597 [0.9458, 0.9736]","0.7653 [0.7361, 0.7958]","0.7461 [0.7247, 0.7683]","0.7265 [0.7042, 0.7476]"
CLAY,"0.7306 [0.6972, 0.7611]","0.9111 [0.8903, 0.9306]","0.9458 [0.9306, 0.9611]","0.7306 [0.6972, 0.7611]","0.6675 [0.6442, 0.6906]","0.6333 [0.6111, 0.6554]"
A-BAP,"0.6667 [0.6347, 0.7000]","0.9014 [0.8792, 0.9222]","0.9472 [0.9305, 0.9625]","0.6667 [0.6347, 0.7000]","0.6933 [0.6689, 0.7181]","0.6975 [0.6742, 0.7201]"


The headline is the **R@1 column**: a clean, CI-separated descent from LDE (0.88) to
A-BAP (0.67). This is the mirror image of CelebA, where A-BAP is the champion (R@1 =
0.1104) and the mined-direction methods (LDE/GDE) sit near the bottom (~0.022).

**Cross-dataset ranking (mean R@1):**

| method | CelebA R@1 | AwA2 R@1 |
|---|---:|---:|
| LDE | 0.0224 | **0.8805** |
| GDE | 0.0221 | 0.8683 |
| CLIP Enhanced | 0.0397 | 0.8083 |
| DGP | 0.0363 | 0.7653 |
| Dual Score | 0.0597 | 0.7569 |
| CLAY | 0.0098 | 0.7306 |
| A-BAP (champion) | **0.1104** | 0.6667 |

> **Scales are not comparable across datasets.** AwA2 is near-ceiling for everyone
> (R@5/R@10 all ≳ 0.90) because its ground truth reduces to *class* retrieval; CelebA's
> per-image rule is far harder. Read the **ordering**, not the magnitudes.

In [3]:
# Headline figure — grouped bars with 95% CI error bars (overlap reads as significance).
out = ROOT / "output" / "awa2" / "visual"; out.mkdir(parents=True, exist_ok=True)
fig = plot_ci_bars(boot, path=out / "awa2_stats_ci_bars.png", title="AwA2 — Recall@K with 95% CI")
fig

<Figure size 900x500 with 1 Axes>

In [4]:
# Recall@K, K = 1…10 — where in the ranking each method's retrieval materializes.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4.5))
for name, d in dumps.items():
    ax.plot(range(1, 11), recall_curve(d, GT_LIST), marker="o", ms=3, label=name)
ax.set_xlabel("K"); ax.set_ylabel("macro Recall@K"); ax.legend(fontsize=7)
ax.set_title("AwA2 — Recall@K, K = 1…10"); fig.tight_layout()
fig.savefig(out / "awa2_stats_recall_curve.png", dpi=120)
fig

<Figure size 700x450 with 1 Axes>

The two panels tell the same story from different angles: at **R@1** the methods
separate cleanly by *how much of their score flows through raw CLIP visual similarity*;
by **R@10** they converge, because on a near-ceiling class-retrieval task the top-10 of
almost any reasonable scorer contains a valid target. The interesting physics is all at
K=1.

## 3. Significance — is the inversion real?

Paired stratified bootstrap of Δ = LDE − model (the champion on AwA2 vs each other
method), Holm–Bonferroni corrected over the {6 metrics × models} family. Same machinery
as the CelebA notebook §B.2. The **CI of Δ** is the significance statement (it and the
p-value cannot contradict by construction).

In [5]:
CHAMP = "LDE"
refs = [n for n in dumps if n != CHAMP]
pvals, diff_rows = {}, []
for ref in refs:
    d = paired_bootstrap_diff(dumps[CHAMP], dumps[ref], B=B, seed=0)
    for f, r in d.items():
        pvals[f"{ref} | {metric_label(f)}"] = r.p_value
        # A difference is significant iff its 95% CI excludes 0 (§A.4) — computed from the
        # numeric bounds, so it can never disagree with the printed CI.
        diff_rows.append({"vs": ref, "metric": metric_label(f), "delta": round(r.delta, 4),
                          "CI95": f"[{r.ci_lo:+.4f}, {r.ci_hi:+.4f}]",
                          "rel_%": round(100 * r.rel_gain, 1), "p_raw": r.p_value,
                          "CI excludes 0": (r.ci_lo > 0) or (r.ci_hi < 0)})
adj = holm_bonferroni(pvals)
dt = pd.DataFrame(diff_rows)
dt["p_holm"] = [adj[f"{r.vs} | {r.metric}"] for r in dt.itertuples()]
dt[dt.metric.isin(["R@1", "R@5", "R@10"])].reset_index(drop=True)

,vs,metric,delta,CI95,rel_%,p_raw,CI excludes 0,p_holm
0,GDE,R@1,0.0125,"[-0.0042, +0.0278]",1.4,0.153846,False,0.307692
1,GDE,R@5,0.0125,"[+0.0028, +0.0222]",1.3,0.019980,True,0.079920
2,GDE,R@10,0.0097,"[+0.0000, +0.0195]",1.0,0.059940,False,0.179820
3,CLIP Enhanced,R@1,0.0722,"[+0.0458, +0.1014]",8.9,0.001998,True,0.059940
4,CLIP Enhanced,R@5,0.0361,"[+0.0194, +0.0514]",3.9,0.001998,True,0.059940
5,CLIP Enhanced,R@10,0.0208,"[+0.0083, +0.0333]",2.2,0.003996,True,0.059940
6,DGP,R@1,0.1153,"[+0.0861, +0.1444]",15.1,0.001998,True,0.059940
7,DGP,R@5,0.0431,"[+0.0250, +0.0625]",4.6,0.001998,True,0.059940
8,DGP,R@10,0.0250,"[+0.0125, +0.0389]",2.6,0.001998,True,0.059940
9,CLAY,R@1,0.1500,"[+0.1208, +0.1819]",20.5,0.001998,True,0.059940


Reading the table:

- **LDE − A-BAP at R@1 = +0.214** with a CI far from zero: the CelebA champion's
  last-place finish on AwA2 is decisive, not sampling noise.
- **LDE − CLIP Enhanced at R@1 = +0.072**, CI excludes 0: the visual-mining advantage
  over text deltas is real.
- **LDE − GDE at R@1**: CI *crosses* zero — LDE and GDE are a statistical **tie** at R@1
  (LDE edges ahead on the denser R@5/P@5/P@10). Honest reading: mined *visual directions*
  win; whether you compose them with flat arithmetic (LDE) or manifold maps (GDE) barely
  matters here.

> **p-value floor.** With only 12 benchmark queries the two-sided bootstrap p cannot fall
> below ≈ 2/(B+1), and Holm over 36 tests lifts every corrected p to that floor (~0.06).
> So read the **Δ confidence intervals** — not the p-values — as the significance
> statement, exactly as the CelebA notebook does with a larger query count. The CIs
> exclude zero for every comparison except LDE−GDE at R@1.

## 4. Why the ranking inverts — the mechanism

This is the part worth understanding. The inversion is not a quirk; it follows
**mathematically** from one structural difference between the datasets.

### 4.1 The datasets differ in *where* attributes are annotated

**CelebA attributes are per-image; AwA2 attributes are class-level.** Every image of an
AwA2 class shares that class's 85-bit row of `predicate-matrix-binary.txt`. This changes
the geometry of the ground-truth set, and therefore *which score function can represent
it*.

**On AwA2 the GT rule collapses to class retrieval.** For a source of class $c_s$, the
valid-target set is the union of images of the few classes $c_t$ that satisfy the ±
constraints and lie within Hamming ≤ 2 of $c_s$. CLIP's visual space clusters animal
photos tightly by species (species are frequent, distinct concepts in CLIP pre-training),
so within-class cosine ≫ between-class cosine. A query $q = v_{\text{ref}} +
\text{small}\cdot\delta$ already ranks the valid set near the top — **the GT-set indicator
is almost a linear functional of the CLIP embedding.** Hence every method scores 0.67+,
and methods are ordered by *how much of their score flows through raw CLIP similarity*.

**On CelebA the GT set is not linearly representable.** Every image is a face — one CLIP
concept — so class-identity margin is ≈ 0; the GT set is a 40-bit per-image pattern
(strict ± bits AND a Hamming ball on ~38 free bits). A single 512-d query scores by one
inner product; a Hamming-ball indicator over ~40 weakly CLIP-encoded bits is far outside
the halfspace family. That is why *all* cosine methods sit at 0.01–0.06 there — and why
A-BAP, which scores the rule itself,
$\;S = \sum_a \log p_a + \sum_b \log(1-p_b) + \log P(\text{Hamming}\le 2)$, wins on CelebA:
its functional form matches the ground truth's generative process.

### 4.2 The CLIP problems each text/probe method inherits

The reason the *same code* that wins on CelebA loses on AwA2 is that AwA2 exposes three
CLIP pathologies which the visual-mining methods (LDE/GDE) sidestep entirely:

1. **Modality gap** (Liang et al. 2022). Text and image embeddings occupy two disjoint
   cones on the sphere. A text delta $\hat t(\text{"horns"})$, even after $\mu_{txt}$
   centering, has only a small component along the *visual* axis separating horned classes
   from the rest. Measured: the text baseline's worst queries are exactly the
   steering-dependent ones — `+horns` 0.633, `+longneck` 0.633 — while identity-dominated
   queries sit at 0.92+. DGP inherits it (0.650/0.667 there): its gate re-weights
   *paraphrases of the same text direction*; it cannot manufacture visual signal.

2. **Non-groundable predicates.** Many of AwA2's 85 predicates (`smart`, `timid`, `fast`,
   `newworld`, ...) are not visible in a single photo. For text methods their prompt
   embeddings are noise; for probe methods (A-BAP) their posteriors are class-prior guesses
   — probe mean accuracy is **86.4%** (min 75%). CelebA's 40 attributes are almost all
   directly visible, so the probes there both score higher (~88.5%) and carry *per-image*
   information no other channel has.

3. **No negation semantics** (Alhamoud et al. 2025, affirmation bias). CLIP's text encoder
   cannot represent "not X", so every method approximates negation geometrically. Which
   approximation wins is dataset-dependent (§4.4).

**A-BAP's collapse, quantified.** Its Poisson-binomial tail runs over $F\approx 80$ free
attributes with per-attribute probe error $\varepsilon\approx 0.14$, so it accumulates an
expected $F\varepsilon \approx 11$ predicted flips — an order of magnitude above the
Hamming radius of 2. On AwA2 the tail term is therefore essentially **probe noise
re-ranking images inside the already-correct class neighbourhood**. On CelebA the same
noisy-but-rule-aware score *beats* clean cosine because there cosine is *adversarial* to
the rule (the `-Male,-Mustache` "female-twin" failure: 0.000 R@K for every geometry tier).
The λ-sweep diagnostic (`src/awa2/tier5_fusion.py --diagnose`) confirms this directly: as
we up-weight a raw-cosine identity term $\lambda\cdot\cos(v_x,v_{\text{ref}})$, A-BAP's R@1
rises monotonically 0.607 → 0.765, converging toward the baseline.

### 4.3 Why the *visual* methods win

**LDE and GDE mine their directions in the visual space**, so they dodge all three
pathologies at once. With class-level labels, the has-$a$ set of attribute $a$ is a union
of whole classes, so the mined direction $v_a = \text{mean}(\text{images with }a) - \mu$
is a contrast of class centroids — *by construction* the axis that separates valid target
classes from the rest, with zero modality gap. This repairs exactly the text methods' weak
queries: `+horns` 0.633 → **0.917**, `+longneck` 0.633 → **0.883**.

### 4.4 Why LDE (flat) beats GDE (manifold)

First, the relationship: **LDE is not an enhancement of GDE — it is the linear special
case GDE generalizes** (Trager et al.'s "ideal words" = GDE with $\mathcal M = \mathbb
R^n$: intrinsic mean → arithmetic mean, Exp/Log → identity). In our code LDE is a zero-diff
ablation of GDE (same mined directions, only the geometric operators differ). So "LDE wins"
means **the ablation won — the spherical machinery did not pay for itself on AwA2.** The
gap decomposes into the two operator differences:

- **Negation: rejection vs subtraction.** GDE removes the *entire* component along
  $\hat v_a$: $q \leftarrow q - (q\cdot\hat v_a)\hat v_a$. For a broad attribute like
  `quadrapedal` (~34 of 50 classes), $v_a$ points near the dominant axis of animal identity
  itself, so the projection is large for every reference and rejection **amputates the
  identity anchor**. LDE's subtraction shifts by the small fixed $\lVert v_a\rVert$ and
  leaves identity intact. Measured: `-quadrapedal` GDE 0.633 vs LDE 0.817 — the single
  query that accounts for most of the mean gap. (CelebA's "negation = no preference on the
  X axis" intuition is right for a *narrow per-image* attribute; wrong for a *broad
  class-level* one entangled with identity.)

- **Curvature: Log/Exp maps vs flat arithmetic.** GDE linearizes at one global $\mu$; the
  log-map factor $\theta/\sin\theta$ amplifies with distance from $\mu$. AwA2 spans 50
  well-separated species clusters (large $\theta$ spread) vs CelebA's single tight "face"
  cluster, so the tangent approximation is stretched exactly where AwA2 lives — while for
  coarse class retrieval flat composition is already sufficient.

## 5. Takeaway

**On AwA2, method quality is monotone in (a) how much of the score flows through raw CLIP
visual similarity and (b) how visually grounded the attribute directions are:** visual
flat (LDE) > visual geodesic (GDE) > text-delta identity (CLIP Enhanced) > text-direction
machinery (DGP, Dual Score, CLAY) > probe-mediated Bayes (A-BAP). On CelebA the ordering
nearly reverses, because there the GT rule is per-image, combinatorial, and
cosine-adversarial — the regime A-BAP was designed for.

This is what a transfer study should produce: not "the best method", but a map of each
method's **inductive bias**. A-BAP's learned rule-scoring needs per-image, visually
groundable attributes and an adversarial cosine; identity-plus-delta arithmetic needs
class-clustered embeddings; visual direction mining needs labels whose granularity (class
vs image) matches the retrieval task. Each is the best method *for the attribute structure
it assumes* — a caveat worth stating explicitly when reporting the CelebA champion.

*Full mechanism write-up:* `notebooks/awa2_negative_transfer_analysis.md`.
*Reproduction:* the runners in `src/awa2/` (see that document's §8).